In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file coontent and metadata
    """

    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")
    
    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()
    
        if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
fabric = read_repo_data('danielmiessler', 'Fabric')

print(f"Fabric documents: {len(fabric)}")

Fabric documents: 395


In [3]:
# Split by chunks
# Eg.: 0-2000, 1000-3000, 2000-4000, etc.

def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    chunks = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        chunks.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break
            
    return chunks

In [4]:
fabric_chunks = []

for doc in fabric:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    # Add the same metadata as the original document to each section
    for chunk in chunks:
        chunk.update(doc_copy)
        
    fabric_chunks.extend(chunks)

In [5]:
print(fabric_chunks[31])

{'start': 30000, 'chunk': '8n keys across all locales, and enforced non-empty validation for Azure deployment names on configure.\n\n## v1.4.415 (2026-02-19)\n\n### PR [#2016](https://github.com/danielmiessler/Fabric/pull/2016) by [ksylvan](https://github.com/ksylvan): Extend Anthropic model beta map with 1M context models\n\n- Extends the Anthropic model beta map to include 1M context window models, adding Claude Sonnet 4.6, Claude Opus 4.5, and Claude Opus 4.6 to the beta entries.\n- Documents 1M token context window model support and clarifies the model beta list maintenance and update strategy.\n- Groups model variants under clearer, annotated sections for improved readability and organization.\n\n## v1.4.414 (2026-02-19)\n\n### PR [#2015](https://github.com/danielmiessler/Fabric/pull/2015) by [ksylvan](https://github.com/ksylvan): Implement comprehensive i18n support across all plugins and tools\n\n- Implement comprehensive internationalization (i18n) support across all AI vendor 

In [6]:
# Split by pargraphs

import re

def split_md_by_paragraphs(text):
    """
    Split markdown files by paragraph
    
    :param text: Markdown text as a string
    :return: List of paragraphs as strings
    """

    # This regex matches markdown paragraphs
    paragraph_pattern = r"\n\s*\n"
    pattern = re.compile(paragraph_pattern, re.MULTILINE)

    # Split into paragraphs
    parts = pattern.split(text)

    paragraphs = []
    for part in parts:
        paragraphs.append(part)

    return paragraphs


In [7]:
fabric_paragraphs = []

for doc in fabric:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    paragraphs = split_md_by_paragraphs(doc_content)

    # Add the same metadata as the original document to each section
    for paragraph in paragraphs:
        paragraph_doc = doc_copy.copy()
        paragraph_doc['paragraph'] = paragraph
        fabric_paragraphs.append(paragraph_doc)

In [8]:
print(fabric_paragraphs[1])

{'filename': 'Fabric-main/.github/pull_request_template.md', 'paragraph': 'Please briefly describe what this PR does.'}


In [9]:
# Split by sections
# Heading 1, Heading 2, etc.

def split_md_by_sections(text, level=2):
    """
    Split markdown text by a specific header level

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """

    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with capturing
        # groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1] # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections

In [10]:
fabric_sections = []

for doc in fabric:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_md_by_sections(doc_content)

    # Add the same metadata as the original document to each section
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        fabric_sections.append(section_doc)

In [11]:
print(fabric_sections[1])

{'filename': 'Fabric-main/.github/pull_request_template.md', 'section': '## Related issues\n\nPlease reference any open issues this PR relates to in here.\nIf it closes an issue, type `closes #[ISSUE_NUMBER]`.'}


In [12]:
# Split using an LLM

from openai import OpenAI

openai_client = OpenAI()

def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {'role': 'user', 'content': prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text

In [13]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

In [14]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document = text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [15]:
from tqdm.auto import tqdm

fabric_llm_chunks = []

# for doc in tqdm(fabric):
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = intelligent_chunking(doc_content)

#     # Add the same metadata as the original document to each section
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         fabric_llm_chunks.append(section_doc)

In [16]:
print(fabric_llm_chunks)

[]


In [23]:
# Index chunks using minsearch

# Text Search
from minsearch import Index

fabric_index = Index(
    text_fields = ['chunk', 'section', 'filename', 'title', 'description'],
    keyword_fields = []
)

# fabric_index.fit(fabric_chunks)
fabric_index.fit(fabric_sections)

# Vector Search
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

# Models
# all-MiniLM-L6-v2 - General purpose, fast and efficient
# all-mpnet-base-v2 - Higher quality, slower
# multi-qa-distilbert-cos-v1 - question-answering tasks

# Turn docs into embeddings

from tqdm.auto import tqdm
import numpy as np

fabric_embeddings = []

for d in tqdm(fabric_sections):
    text = d['section']
    v = embedding_model.encode(text)
    fabric_embeddings.append(v)

fabric_embeddings = np.array(fabric_embeddings)

from minsearch import VectorSearch

# Create a vector search index using our embeddings and original documents
fabric_vindex = VectorSearch()
fabric_vindex.fit(fabric_embeddings, fabric_sections)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/1230 [00:00<?, ?it/s]

In [31]:
# Search index

def text_search(query):
    return fabric_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return fabric_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results

In [36]:
query = 'What pattern can I use to summarize?'
results = vector_search(query)

In [37]:
print(results)

[{'filename': 'Fabric-main/data/patterns/write_hackerone_report/system.md', 'section': '## Summary:'}, {'filename': 'Fabric-main/data/patterns/write_hackerone_report/system.md', 'section': '## Summary:'}, {'filename': 'Fabric-main/data/patterns/suggest_pattern/user.md', 'section': "## SUMMARIZATION PATTERNS\n\n### capture_thinkers_work\n\nExtract key concepts, background, and ideas from notable thinkers' work.\n\n### create_5_sentence_summary\n\nGenerate concise summaries of content in five levels, five words to one.\n\n### create_micro_summary\n\nGenerate concise summaries with one-sentence overview and key points.\n\n### create_summary\n\nGenerate concise summaries by extracting key points and main ideas.\n\n### summarize\n\nGenerate summaries capturing key points and details.\n\n### summarize_debate\n\nSummarize debates highlighting arguments and agreements.\n\n### summarize_lecture\n\nSummarize lectures capturing key concepts and takeaways.\n\n### summarize_legislation\n\nSummarize